# 🏁 Notebook 4: Concurrent Retries & Race Conditions

What if a client retries *before* the first request has finished? Both requests arrive at the server at essentially the same time, each carrying the same idempotency key. Naively, they'll both read the idempotency table, see nothing, and each charge the customer.

Real systems solve this with two tools:

1. **An atomic `INSERT`** with a `UNIQUE` constraint — only one inserter wins.
2. **An `IN_PROGRESS` marker** so a concurrent retry can wait/reject instead of running the side effect again.

We'll demonstrate the problem first, then the fix.

## 🛠️ Setup

```bash
cd 04-patterns/idempotency
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

## 🟥 BAD: check-then-act race

Two threads interleave between *read* and *write* — both think they're first.

In [ ]:
import threading, time, uuid

class RacyService:
    def __init__(self):
        self.balances = {'alice': 100}
        self.idem = {}

    def charge(self, key, amount):
        if key in self.idem:
            return ('REPLAY', self.idem[key])
        # simulate some work — this is where the window opens for a race
        time.sleep(0.05)
        self.balances['alice'] -= amount
        result = {'balance': self.balances['alice']}
        self.idem[key] = result
        return ('FRESH', result)

svc = RacyService()
k = str(uuid.uuid4())
results = []

def worker():
    results.append(svc.charge(k, 10))

threads = [threading.Thread(target=worker) for _ in range(5)]
for t in threads: t.start()
for t in threads: t.join()

print('results    :', results)
print('final bal  :', svc.balances['alice'], '← charged multiple times for ONE logical request 😱')


Five concurrent retries, five charges. The `if key in self.idem` check is too late — all threads passed it before any thread had finished writing.

## 🟩 GOOD: atomic INSERT wins the race

Instead of *check-then-act*, attempt an atomic `INSERT` of an `IN_PROGRESS` marker keyed by the idempotency key. The `PRIMARY KEY` constraint means **at most one insert can succeed** — the winner runs the side effect, the losers wait for its result.

> ⚠️ One detail that is easy to get wrong and expensive to debug: **each thread needs its own database connection.** A single `sqlite3.Connection` shared across threads has one transaction state, so a reader in one thread can disturb a writer in another — and this demo will silently double-charge every few runs while still printing a ✅. Real servers use a connection pool; we do the same below.

In [ ]:
import sqlite3, json, os, tempfile

# A real server has a CONNECTION POOL, not one shared connection. That matters here:
# a single sqlite3.Connection shared across threads has one transaction state, so a
# reader in thread B can interfere with a writer in thread A — and the "fix" below
# would quietly double-charge every few runs. One connection per thread, and we let
# the DATABASE do the serializing. (An in-memory DB is private per connection, so we
# need a file.)
DB_PATH = os.path.join(tempfile.mkdtemp(), 'bank.db')
_boot = sqlite3.connect(DB_PATH)
_boot.executescript('''
PRAGMA journal_mode=WAL;              -- concurrent readers alongside one writer
CREATE TABLE accounts (name TEXT PRIMARY KEY, balance INTEGER);
CREATE TABLE idem (
    key         TEXT PRIMARY KEY,     -- <- THIS is what makes the claim atomic
    status      TEXT NOT NULL,        -- 'IN_PROGRESS' | 'DONE'
    result_json TEXT
);
INSERT INTO accounts VALUES ('alice', 100);
''')
_boot.commit(); _boot.close()

_local = threading.local()

def db():
    """One connection per thread — the pool a real web server would hand you."""
    if not hasattr(_local, 'conn'):
        conn = sqlite3.connect(DB_PATH, isolation_level=None)  # we manage transactions
        conn.execute('PRAGMA busy_timeout=5000')               # wait, don't fail, on lock
        _local.conn = conn
    return _local.conn


def safe_charge(key, amount):
    conn = db()

    # 1. CLAIM: one atomic INSERT. The PRIMARY KEY decides the winner — not a lock,
    #    not a check-then-act. Exactly one of N concurrent inserts can succeed.
    try:
        conn.execute("INSERT INTO idem(key, status) VALUES (?, 'IN_PROGRESS')", (key,))
        claimed = True
    except sqlite3.IntegrityError:
        claimed = False

    # 2. LOSER: someone else is doing the work. Wait for their result and replay it.
    #    (Returning 409 immediately is the other valid answer — see the note below.)
    if not claimed:
        for _ in range(200):
            row = conn.execute('SELECT status, result_json FROM idem WHERE key=?',
                               (key,)).fetchone()
            if row and row[0] == 'DONE':
                return ('REPLAY', json.loads(row[1]))
            time.sleep(0.005)
        return ('409_IN_PROGRESS', None)

    # 3. WINNER: do the work, then publish the result — in ONE transaction, so a
    #    crash can't leave the money moved with no record of it (notebook 3).
    try:
        conn.execute('BEGIN IMMEDIATE')
        conn.execute("UPDATE accounts SET balance = balance - ? WHERE name='alice'", (amount,))
        bal = conn.execute("SELECT balance FROM accounts WHERE name='alice'").fetchone()[0]
        result = {'balance': bal}
        conn.execute("UPDATE idem SET status='DONE', result_json=? WHERE key=?",
                     (json.dumps(result), key))
        conn.execute('COMMIT')
        return ('FRESH', result)
    except Exception:
        conn.execute('ROLLBACK')
        # Release the claim so a later retry isn't locked out. (Not enough on its
        # own — if the PROCESS dies we never get here. That's the next section.)
        conn.execute("DELETE FROM idem WHERE key=? AND status='IN_PROGRESS'", (key,))
        raise


k = str(uuid.uuid4())
results = []
results_lock = threading.Lock()

def worker():
    r = safe_charge(k, 10)
    with results_lock:
        results.append(r)

threads = [threading.Thread(target=worker) for _ in range(5)]
for t in threads: t.start()
for t in threads: t.join()

fresh = sum(1 for r in results if r[0] == 'FRESH')
bal = db().execute("SELECT balance FROM accounts WHERE name='alice'").fetchone()[0]
print('FRESH count:', fresh, ' REPLAY count:', sum(1 for r in results if r[0] == 'REPLAY'))
print('final bal  :', bal)

# Don't take the checkmark's word for it — assert it, so this cell fails loudly
# if the concurrency ever regresses.
assert fresh == 1, f'expected exactly one winner, got {fresh}'
assert bal == 90, f'expected exactly one $10 charge, balance is {bal}'
print('exactly ONE charge applied, from 5 simultaneous identical requests')

## 🟥 The failure the claim itself introduces: a wedged key

Look again at what happens when the winner dies *after* claiming the key but *before* writing `DONE` — a process kill, an OOM, a pod eviction. The `IN_PROGRESS` row survives, and nothing will ever set it to `DONE`.

Every future retry of that key now takes the `not claimed` branch, polls, times out, and gives up. The customer's charge is stuck forever, and the only cure is a human with database access. We traded a double-charge for a permanent wedge — which is better, but it isn't done.

In [ ]:
# Wedge a key exactly the way a killed process would: claim it, then die.
def crashing_charge(key, amount):
    db().execute("INSERT INTO idem(key, status) VALUES (?, 'IN_PROGRESS')", (key,))
    raise SystemExit('process killed after claiming the key, before writing DONE')

wedged = str(uuid.uuid4())
try:
    crashing_charge(wedged, 10)
except SystemExit as e:
    print('crash:', e)

print('row left behind:', db().execute(
    'SELECT key, status FROM idem WHERE key=?', (wedged,)).fetchone())

# A retry — a fresh request, minutes later — can never make progress.
t0 = time.monotonic()
print('later retry    :', safe_charge(wedged, 10),
      f'(after waiting {time.monotonic() - t0:.1f}s)')
print('...and that is the answer forever. The charge is stuck until a human')
print('   opens a psql prompt. We traded a double-charge for a permanent wedge.')

## 🟩 Fix: the claim is a *lease*, not a lock

Give the claim an expiry. A claim older than the lease is assumed dead and may be taken over. This is the same idea as a queue's *visibility timeout* (SQS) or a lock's TTL (Redis `SET NX PX`) — you never take a lock you can't reclaim.

Two things make this safe:

1. **The takeover must itself be atomic** — a conditional `UPDATE` that only succeeds if the row still looks expired, so two rescuers can't both take over.
2. **The lease must be longer than the work.** If the work can take 30s, a 5s lease means a *healthy* worker gets its claim stolen and you're back to two concurrent executions. Lease > p99 work time, with room to spare — and renew it (a *heartbeat*) if the work can run long.

In [ ]:
LEASE_SECONDS = 0.5      # in production: comfortably longer than p99 work time

db().execute('ALTER TABLE idem ADD COLUMN claimed_at REAL')   # when the claim was taken

def leased_charge(key, amount, lease=LEASE_SECONDS):
    conn, now = db(), time.time()
    try:
        conn.execute("INSERT INTO idem(key, status, claimed_at) VALUES (?, 'IN_PROGRESS', ?)",
                     (key, now))
        claimed = True
    except sqlite3.IntegrityError:
        # The key exists. Take it over ONLY if its lease expired — and do that with a
        # conditional UPDATE, so if two rescuers race, exactly one sees rowcount == 1.
        cur = conn.execute(
            "UPDATE idem SET claimed_at=? "
            "WHERE key=? AND status='IN_PROGRESS' AND claimed_at < ?",
            (now, key, now - lease))
        claimed = cur.rowcount == 1
        if claimed:
            print(f'  lease expired -> taking over {key[:8]}...')

    if not claimed:
        row = conn.execute('SELECT status, result_json FROM idem WHERE key=?',
                           (key,)).fetchone()
        if row and row[0] == 'DONE':
            return ('REPLAY', json.loads(row[1]))
        return ('409_IN_PROGRESS', None)     # original is genuinely still running

    conn.execute('BEGIN IMMEDIATE')
    conn.execute("UPDATE accounts SET balance = balance - ? WHERE name='alice'", (amount,))
    bal = conn.execute("SELECT balance FROM accounts WHERE name='alice'").fetchone()[0]
    conn.execute("UPDATE idem SET status='DONE', result_json=? WHERE key=?",
                 (json.dumps({'balance': bal}), key))
    conn.execute('COMMIT')
    return ('FRESH', {'balance': bal})

# Wedge a fresh key the same way the crash did, then let a later retry rescue it.
wedged2 = str(uuid.uuid4())
db().execute("INSERT INTO idem(key, status, claimed_at) VALUES (?, 'IN_PROGRESS', ?)",
             (wedged2, time.time()))

before = db().execute("SELECT balance FROM accounts WHERE name='alice'").fetchone()[0]
print('retry while the lease is valid:', leased_charge(wedged2, 10), '<- correctly refused')
time.sleep(LEASE_SECONDS + 0.05)
print('retry after the lease expires :', leased_charge(wedged2, 10))
print('replay of that same key       :', leased_charge(wedged2, 10))
after = db().execute("SELECT balance FROM accounts WHERE name='alice'").fetchone()[0]

assert after == before - 10, 'the rescued request must charge exactly once'
print(f'\nbalance {before} -> {after}: charged exactly once, despite a crash mid-request.')

> ⚠️ **The honest caveat.** A lease bounds how long a key stays wedged; it does **not** prove the dead worker's side effect never happened. If the process died *after* the card was charged but before the commit, the takeover will charge again. Inside one database you avoid this by putting the side effect and the record in one transaction (notebook 3). Across a network boundary — a payment gateway, an email provider — you can't, so you rely on **the downstream's own idempotency key**: pass your key through to Stripe, and *their* dedup covers the gap yours can't.

## 🧠 Takeaways

- *Check-then-act* is broken under concurrency. Make the claim **atomic** — one `INSERT` with a `UNIQUE` constraint, not a read followed by a write.
- An `IN_PROGRESS` marker lets a concurrent retry **wait** for the original, or fail fast with `409`. Returning `409` is usually kinder than making the client hold a connection open.
- **A claim needs an expiry.** Without a lease, one crashed worker wedges that key forever.
- Freeing the claim on failure is not optional, and `try/finally` is not enough — the process can die between the two. That's what the lease is for.

## 🧭 When to use which mechanism

| Situation | Reach for |
|---|---|
| The operation names an **end state** (`set status`, `PUT` a full resource) | Nothing. It's naturally idempotent (notebook 1). |
| A **single-database** side effect (insert a row, move a balance) | The side effect + the key row in **one transaction** (notebook 3). Simplest thing that is actually correct. |
| An **external** side effect (charge a card, send an email, call a partner API) | A key store *plus* pass your key to the downstream so its dedup covers your crash window. |
| Concurrent retries are realistic (mobile clients, aggressive proxies, queue redelivery) | Atomic claim + `IN_PROGRESS` + **lease** (this notebook). |
| A **message consumer** that can be redelivered | Usually the same pattern, keyed on the message id — but check whether the broker already offers it (SQS FIFO dedup, Kafka `enable.idempotence`) before you build it. |

## 🚫 When you don't need any of this

- **Reads.** A `GET` is already idempotent; a dedup store just adds a cache you have to invalidate.
- **Operations that are cheap and harmless to repeat** — recomputing a derived value, re-uploading the same blob to the same key, re-running a `DELETE`.
- **When the natural rewrite is available.** A `UNIQUE` constraint on a client-supplied `order_id` with `ON CONFLICT DO NOTHING` is idempotency, costs one index, and has no TTL, no lease, and no cleanup job. Prefer it whenever the data model allows.

## ⚖️ What this costs you

- **A write on the hot path.** Every request now does an extra `INSERT`, and it's the *first* thing it does, so it's on the critical path of your p99.
- **A row per request, times the retention window.** At 1k req/s and 24h TTL that's ~86M rows to store and purge. The purge job is not optional and it is not free.
- **A new failure mode.** The dedup store is now a hard dependency. Failing open means duplicates; failing closed means an outage. Decide which, deliberately, before it happens.
- **A contract with clients.** Once you accept `Idempotency-Key`, its semantics (scope, TTL, what happens on `5xx`) are public API. Document them.